# Moodify – Notebook 2/2: Mô hình tiếng Việt (PhoBERT, 6 cảm xúc)

**Cần có trong `MyDrive/train-KLCN/`:** `moodify_emotion_en.csv` (do Notebook 1 tạo), `uit_vsmec.csv`, `vietnamese_emoji.csv`.

**Quy trình**
1. Dịch tập `dair-ai/emotion` (đã chia train/val/test ở Notebook 1) sang tiếng Việt bằng `vinai/vinai-translate-en2vi-v2` (có lưu cache từng đợt, mất kết nối vẫn chạy tiếp được).
2. Thêm 5 lớp tương ứng của UIT-VSMEC (bỏ Disgust, Other) làm dữ liệu tiếng Việt thật: có teencode, emoji, thiếu dấu.
3. Huấn luyện PhoBERT với tiền xử lý emoji → chữ + PyVi như cũ, **nhưng thay từ điển teencode tự sinh bằng bảng teencode nhỏ viết tay** (xem mục 2 vì sao).
4. Đánh giá **tách riêng** từng tập test: (a) test dịch máy (số để so với mục tiêu 85%), (b) test UIT-VSMEC 5 lớp (số phản ánh tiếng Việt MXH thật), (c) tập người gán nhãn của Moodify nếu bạn có.

⚠️ Số ở (a) là chuẩn **trong cùng phân phối với dữ liệu train**, chưa chứng minh chất lượng trên câu người dùng thật. Số ở (b) và (c) mới là bằng chứng thực tế.

In [ ]:
!pip install -q transformers datasets sentencepiece pyvi scikit-learn pandas

In [ ]:
import os, re, json, math, random, datetime, logging, unicodedata
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM,
                          get_linear_schedule_with_warmup)
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from pyvi import ViTokenizer

# ======================= CẤU HÌNH =======================
SEED = 42
DRIVE_PATH = "/content/drive/MyDrive/train-KLCN"

MODEL_NAME = "vinai/phobert-base-v2"   # giữ PhoBERT để khớp service hiện tại.
# Thử thay bằng "uitnlp/visobert" (SOTA UIT-VSMEC, chịu emoji/teencode/thiếu dấu tốt hơn).
# Khi đó service cũng phải đổi theo: không còn tách từ bằng PyVi. (Chưa chạy thử; ViSoBERT chỉ cho mục đích nghiên cứu.)
USE_WORD_SEGMENTATION = MODEL_NAME.startswith("vinai/phobert")
USE_EMOJI_DICT = USE_WORD_SEGMENTATION       # ViSoBERT đã hiểu emoji
USE_TEENCODE = USE_WORD_SEGMENTATION

MAX_LEN = 96
BATCH_SIZE = 32
EPOCHS = 5
LEARNING_RATE = 3e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
PATIENCE = 2
TARGET_ACCURACY = 0.85

USE_VSMEC_AUX = True          # thêm 5 lớp UIT-VSMEC vào train/val
NO_DIACRITIC_AUG_RATE = 0.10  # 10% câu train được nhân bản dạng không dấu (PhoBERT rất yếu với text không dấu)

# Dịch máy
MT_NAME = "vinai/vinai-translate-en2vi-v2"   # giấy phép AGPL-3.0
MT_BATCH = 48
MT_CHUNK = 2000
MT_BEAMS = 4

LABELS = ["Sadness", "Enjoyment", "Love", "Anger", "Fear", "Surprise"]   # phải trùng Notebook 1
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}
VA_ANCHORS = {
    "Sadness":   [-0.8, -0.6],
    "Enjoyment": [ 0.8,  0.7],
    "Love":      [ 0.8,  0.3],   # Love là giá trị mới (đề xuất), chỉnh theo vùng mood trong DB
    "Anger":     [-0.8,  0.8],
    "Fear":      [-0.7,  0.9],
    "Surprise":  [ 0.5,  0.9],
}
VSMEC_MAP = {"sadness": "Sadness", "enjoyment": "Enjoyment", "anger": "Anger",
             "fear": "Fear", "surprise": "Surprise"}      # Disgust, Other bị bỏ có chủ đích
# ========================================================

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    DRIVE_PATH = "./train-KLCN"
os.makedirs(DRIVE_PATH, exist_ok=True)

EN_CSV = os.path.join(DRIVE_PATH, "moodify_emotion_en.csv")
VI_CSV = os.path.join(DRIVE_PATH, "moodify_emotion_vi.csv")
VSMEC_CSV = os.path.join(DRIVE_PATH, "uit_vsmec.csv")
EMOJI_CSV = os.path.join(DRIVE_PATH, "vietnamese_emoji.csv")
HUMAN_TEST_CSV = os.path.join(DRIVE_PATH, "moodify_vi_human_test.csv")   # tùy chọn: cột text,label
CACHE_DIR = os.path.join(DRIVE_PATH, "translation_cache")
os.makedirs(CACHE_DIR, exist_ok=True)

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = os.path.join(DRIVE_PATH, f"output_vi_emotion_{timestamp}")
os.makedirs(OUTPUT_DIR, exist_ok=True)

for h in logging.root.handlers[:]:
    logging.root.removeHandler(h)
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.FileHandler(os.path.join(OUTPUT_DIR, "training_log.txt"), mode="w", encoding="utf-8"),
              logging.StreamHandler()])
for noisy in ("httpx", "httpcore", "huggingface_hub", "urllib3", "filelock"):
    logging.getLogger(noisy).setLevel(logging.WARNING)
logger = logging.getLogger("moodify_vi")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
logger.info("Device: %s | AMP: %s | Output: %s", device, use_amp, OUTPUT_DIR)
if device.type != "cuda":
    logger.warning("Đang chạy CPU - dịch và train sẽ rất chậm. Hãy bật GPU trong Colab.")
if not os.path.isfile(EN_CSV):
    raise FileNotFoundError(f"Chưa có {EN_CSV}. Hãy chạy Notebook 1 trước.")

## 1. Dịch `dair-ai/emotion` sang tiếng Việt

Tiền xử lý trước khi dịch: viết hoa chữ đầu và thêm dấu chấm cuối (dữ liệu gốc toàn chữ thường, không dấu câu → MT dịch tệ hơn).
Dịch theo đợt `MT_CHUNK` câu, mỗi đợt lưu cache; nếu Colab ngắt kết nối, chạy lại cell này sẽ tiếp tục từ đợt dở.
**Thời gian:** phụ thuộc số câu và GPU (mình chưa đo trên T4 của bạn). Với PER_CLASS_TARGET=4000 là khoảng 30k câu.

In [ ]:
if os.path.isfile(VI_CSV):
    logger.info("Đã có %s - bỏ qua bước dịch. Xóa file này nếu muốn dịch lại.", VI_CSV)
else:
    tok_t = AutoTokenizer.from_pretrained(MT_NAME, src_lang="en_XX")
    mt = AutoModelForSeq2SeqLM.from_pretrained(MT_NAME).to(device)
    if device.type == "cuda":
        mt = mt.half()
    mt.eval()
    VI_LANG_ID = tok_t.convert_tokens_to_ids("vi_VN")
    assert VI_LANG_ID is not None and VI_LANG_ID != tok_t.unk_token_id, "Không tìm thấy token ngôn ngữ vi_VN"

    def prepare_for_mt(t):
        t = str(t).strip()
        if not t:
            return t
        t = t[0].upper() + t[1:]
        return t if t[-1] in ".!?" else t + "."

    @torch.no_grad()
    def translate_batch(texts):
        enc = tok_t([prepare_for_mt(t) for t in texts], return_tensors="pt", padding=True,
                    truncation=True, max_length=128).to(device)
        out = mt.generate(**enc, decoder_start_token_id=VI_LANG_ID, num_beams=MT_BEAMS,
                          max_new_tokens=128, early_stopping=True)
        return [s.strip() for s in tok_t.batch_decode(out, skip_special_tokens=True)]

    df_en = pd.read_csv(EN_CSV, encoding="utf-8-sig")
    unique_texts = sorted(df_en["text"].astype(str).unique(), key=lambda t: (len(t), t))
    n_chunks = math.ceil(len(unique_texts) / MT_CHUNK)
    logger.info("Cần dịch %d câu duy nhất (%d đợt).", len(unique_texts), n_chunks)

    for ci in range(n_chunks):
        path = os.path.join(CACHE_DIR, f"chunk_{ci:04d}.csv")
        if os.path.isfile(path):
            continue
        chunk = unique_texts[ci * MT_CHUNK:(ci + 1) * MT_CHUNK]
        outs = []
        for i in range(0, len(chunk), MT_BATCH):
            outs += translate_batch(chunk[i:i + MT_BATCH])
        pd.DataFrame({"text_en": chunk, "text": outs}).to_csv(path, index=False, encoding="utf-8-sig")
        logger.info("Dịch xong đợt %d/%d", ci + 1, n_chunks)

    tr = pd.concat([pd.read_csv(os.path.join(CACHE_DIR, f"chunk_{ci:04d}.csv"), encoding="utf-8-sig")
                    for ci in range(n_chunks)], ignore_index=True).fillna("")
    mapping = dict(zip(tr["text_en"], tr["text"]))
    df_vi = df_en.copy()
    df_vi["text_en"] = df_vi["text"].astype(str)
    df_vi["text"] = df_vi["text_en"].map(mapping).fillna("")

    def good_translation(src, tgt):
        tgt = str(tgt).strip()
        if len(tgt) < 3 or tgt.lower() == str(src).strip().lower():
            return False
        return 0.4 <= len(tgt) / max(len(str(src)), 1) <= 3.0

    ok = np.array([good_translation(s, t) for s, t in zip(df_vi["text_en"], df_vi["text"])])
    logger.info("Loại %d/%d câu dịch lỗi (rỗng / không dịch / độ dài bất thường).", int((~ok).sum()), len(ok))
    df_vi = df_vi[ok].reset_index(drop=True)
    df_vi.to_csv(VI_CSV, index=False, encoding="utf-8-sig")
    logger.info("Đã lưu %s (%d câu).", VI_CSV, len(df_vi))
    del mt
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

df_vi = pd.read_csv(VI_CSV, encoding="utf-8-sig")
print(df_vi.groupby("split")["label_name"].value_counts().unstack(0))
print("\nVí dụ (hãy ĐỌC thử ~30 dòng để chắc bản dịch giữ đúng cảm xúc):")
print(df_vi.sample(8, random_state=SEED)[["text_en", "text", "label_name"]].to_string(index=False))

## 2. Tiền xử lý + dữ liệu UIT-VSMEC

⚠️ **Lỗi phát hiện trong pipeline cũ:** `vietnamese_teencode.csv` không phải teencode thật mà là cặp (câu đúng, câu bị nhiễu ký tự ngẫu nhiên). Từ điển 3.320 mục tự trích từ file đó biến **~10% token của UIT-VSMEC** thành từ sai, ví dụ `thôi→hôi`, `tôi→ôi`, `sao→ao`, `chưa→trưa`, `thế→hế`, `phim→phím`, `chuyện→truyện`, `con→còn`. Notebook này thay bằng bảng viết tay `CURATED_TEENCODE`.

Phần emoji → chữ và PyVi giữ nguyên logic cũ (bước emoji bỏ qua nhanh khi câu không có emoji, kết quả không đổi).

In [ ]:
def load_emoji_dictionary(path):
    d = pd.read_csv(path, encoding="utf-8-sig")
    res = {}
    for e, desc in zip(d["emoji"], d["description"]):
        if pd.isna(e) or pd.isna(desc):
            continue
        e, desc = str(e).strip(), str(desc).strip()
        if e and desc:
            res[e] = desc
    return dict(sorted(res.items(), key=lambda kv: len(kv[0]), reverse=True))

# Bảng teencode viết tay, chỉ gồm các viết tắt phổ biến và KHÔNG gây nhầm với từ đúng.
# Cố ý không chuẩn hóa từ cảm xúc/chửi thề (vcl, đm, vl...) vì chính chúng mang tín hiệu cảm xúc.
CURATED_TEENCODE = {
    "ko": "không", "k": "không", "kh": "không", "hk": "không", "hok": "không", "khg": "không",
    "dc": "được", "đc": "được", "dk": "được", "vs": "với", "j": "gì", "ntn": "như thế nào",
    "r": "rồi", "rùi": "rồi", "roi": "rồi", "oy": "rồi", "z": "vậy", "zậy": "vậy", "v": "vậy",
    "bn": "bao nhiêu", "mn": "mọi người", "bt": "bình thường", "mk": "mình", "ng": "người", "nta": "người ta",
    "cx": "cũng", "cg": "cũng", "wa": "quá", "wá": "quá", "qá": "quá",
    "thik": "thích", "thix": "thích", "iu": "yêu", "bik": "biết", "bít": "biết", "bjt": "biết",
    "wen": "quên", "qen": "quên", "hnay": "hôm nay", "hqua": "hôm qua", "ae": "anh em", "trc": "trước",
}

EMOJI_DICT = load_emoji_dictionary(EMOJI_CSV) if (USE_EMOJI_DICT and os.path.isfile(EMOJI_CSV)) else {}
TEENCODE_DICT = dict(CURATED_TEENCODE) if USE_TEENCODE else {}
logger.info("Emoji: %d | Teencode: %d | Tách từ PyVi: %s", len(EMOJI_DICT), len(TEENCODE_DICT), USE_WORD_SEGMENTATION)

def preprocess_vietnamese_text(text):
    if not isinstance(text, str) or not text.strip():
        return ""
    if EMOJI_DICT and any(ord(c) > 0x2000 for c in text):
        for emo, desc in EMOJI_DICT.items():
            if emo in text:
                text = text.replace(emo, f" {desc} ")
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    words = text.lower().split()
    if TEENCODE_DICT:
        words = [TEENCODE_DICT.get(w, w) for w in words]
    text = re.sub(r"\s+", " ", " ".join(words)).strip()
    if not text:
        return ""
    return ViTokenizer.tokenize(text).lower() if USE_WORD_SEGMENTATION else text

def strip_diacritics(text):
    text = text.replace("đ", "d").replace("Đ", "D")
    return "".join(c for c in unicodedata.normalize("NFD", text) if unicodedata.category(c) != "Mn")

In [ ]:
def to_frame(texts, labels, source):
    return pd.DataFrame({"text": list(texts), "label": [int(x) for x in labels], "source": source})

mt_train = df_vi[df_vi["split"] == "train"]
mt_val = df_vi[df_vi["split"] == "val"]
mt_test = df_vi[df_vi["split"] == "test"]
train_raw = to_frame(mt_train["text"], mt_train["label"], "mt")
val_raw = to_frame(mt_val["text"], mt_val["label"], "mt")
test_mt_raw = to_frame(mt_test["text"], mt_test["label"], "mt")
test_vsmec_raw = None

if USE_VSMEC_AUX:
    vs = pd.read_csv(VSMEC_CSV, encoding="utf-8-sig")[["Sentence", "Emotion"]].dropna()
    assert len(vs) == 6927, f"UIT-VSMEC phải có 6927 dòng, hiện có {len(vs)} (file bị lọc/đổi thứ tự?)"
    # Khôi phục split chính thức theo thứ tự concat train -> validation -> test trong notebook tải dữ liệu
    parts = {"train": vs.iloc[:5548], "val": vs.iloc[5548:6234], "test": vs.iloc[6234:]}
    def vs_frame(p):
        p = p.copy()
        p["Emotion"] = p["Emotion"].astype(str).str.strip().str.lower().map(VSMEC_MAP)
        p = p.dropna(subset=["Emotion"])
        return to_frame(p["Sentence"], p["Emotion"].map(LABEL2ID), "vsmec")
    train_raw = pd.concat([train_raw, vs_frame(parts["train"])], ignore_index=True)
    val_raw = pd.concat([val_raw, vs_frame(parts["val"])], ignore_index=True)
    test_vsmec_raw = vs_frame(parts["test"])
    logger.info("VSMEC (5 lớp): train +%d | val +%d | test %d", len(vs_frame(parts["train"])),
                len(vs_frame(parts["val"])), len(test_vsmec_raw))

# Nhân bản ngẫu nhiên một phần câu train ở dạng KHÔNG DẤU
if NO_DIACRITIC_AUG_RATE > 0:
    aug = train_raw.sample(frac=NO_DIACRITIC_AUG_RATE, random_state=SEED).copy()
    aug["text"] = aug["text"].map(strip_diacritics)
    aug["source"] = "aug_nodiac"
    train_raw = pd.concat([train_raw, aug], ignore_index=True)

def finalize(df):
    df = df.copy()
    df["x"] = df["text"].map(preprocess_vietnamese_text)
    return df[df["x"].str.strip() != ""].reset_index(drop=True)

train_df = finalize(train_raw).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
val_df = finalize(val_raw)
test_mt_df = finalize(test_mt_raw)
test_vsmec_df = finalize(test_vsmec_raw) if test_vsmec_raw is not None else None

logger.info("Sau tiền xử lý: train=%d | val=%d | test_mt=%d | test_vsmec=%s",
            len(train_df), len(val_df), len(test_mt_df), None if test_vsmec_df is None else len(test_vsmec_df))
logger.info("Nguồn train: %s", train_df["source"].value_counts().to_dict())
logger.info("Nhãn train : %s", train_df["label"].map(ID2LABEL).value_counts().to_dict())

## 3. Huấn luyện PhoBERT

In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts, self.labels = list(texts), [int(x) for x in labels]
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, i):
        return self.texts[i], self.labels[i]

def make_loader(df, tokenizer, shuffle):
    def collate(batch):
        texts, labels = zip(*batch)
        enc = tokenizer(list(texts), padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt")
        enc["labels"] = torch.tensor(labels, dtype=torch.long)
        return enc
    return DataLoader(TextDataset(df["x"], df["label"]), batch_size=BATCH_SIZE,
                      shuffle=shuffle, collate_fn=collate, num_workers=0)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    ys, ps, confs, loss_sum, n = [], [], [], 0.0, 0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            out = model(**batch)
        probs = torch.softmax(out.logits.float(), dim=-1)
        conf, pred = probs.max(dim=-1)
        bs = batch["labels"].size(0)
        loss_sum += out.loss.item() * bs; n += bs
        ys += batch["labels"].tolist(); ps += pred.tolist(); confs += conf.tolist()
    return {"loss": loss_sum / max(n, 1), "accuracy": accuracy_score(ys, ps),
            "macro_f1": f1_score(ys, ps, average="macro", zero_division=0),
            "labels": ys, "preds": ps, "confs": confs}

def run_training(model, tokenizer, train_loader, val_loader, out_dir):
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, max(1, int(total_steps * WARMUP_RATIO)), total_steps)
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)
    best_dir = os.path.join(out_dir, "best_model")
    best_acc, best_f1, bad, history = -1.0, -1.0, 0, []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        loss_sum, correct, seen = 0.0, 0, 0
        for step, batch in enumerate(train_loader, 1):
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                out = model(**batch)
            loss = out.loss
            if not torch.isfinite(loss):
                raise RuntimeError(f"Loss không hữu hạn ở epoch {epoch}, step {step}")
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); scheduler.step()
            bs = batch["labels"].size(0)
            loss_sum += loss.item() * bs; seen += bs
            correct += (out.logits.argmax(-1) == batch["labels"]).sum().item()
            if step % 100 == 0 or step == len(train_loader):
                logger.info("Epoch %d/%d step %d/%d | loss %.4f | acc %.4f",
                            epoch, EPOCHS, step, len(train_loader), loss_sum / seen, correct / seen)

        val = evaluate(model, val_loader)
        history.append({"epoch": epoch, "train_loss": loss_sum / seen, "train_acc": correct / seen,
                        "val_loss": val["loss"], "val_acc": val["accuracy"], "val_macro_f1": val["macro_f1"]})
        logger.info("== Epoch %d | val loss %.4f | val acc %.4f | val macro-F1 %.4f",
                    epoch, val["loss"], val["accuracy"], val["macro_f1"])
        improved = val["accuracy"] > best_acc or (
            math.isclose(val["accuracy"], best_acc, abs_tol=1e-12) and val["macro_f1"] > best_f1)
        if improved:
            best_acc, best_f1, bad = val["accuracy"], val["macro_f1"], 0
            model.save_pretrained(best_dir); tokenizer.save_pretrained(best_dir)
            logger.info("Lưu checkpoint tốt nhất (epoch %d, val acc %.4f)", epoch, best_acc)
        else:
            bad += 1
            if bad >= PATIENCE:
                logger.info("Early stopping sau epoch %d.", epoch)
                break
        with open(os.path.join(out_dir, "training_metrics.json"), "w", encoding="utf-8") as f:
            json.dump(history, f, ensure_ascii=False, indent=2)
    return best_dir, history

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_loader = make_loader(train_df, tokenizer, shuffle=True)
val_loader = make_loader(val_df, tokenizer, shuffle=False)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID).to(device)

best_dir, history = run_training(model, tokenizer, train_loader, val_loader, OUTPUT_DIR)
logger.info("Train xong. Best checkpoint: %s", best_dir)

## 4. Đánh giá: ba tập test, báo cáo riêng, không gộp số

| Tập | Ý nghĩa |
|---|---|
| `test_mt` | Câu dịch máy từ test chính thức của dair-ai. **Số để so với 85%**, nhưng cùng phân phối với train. |
| `test_vsmec_5cls` | Câu tiếng Việt thật trên MXH (5 lớp). Phản ánh độ khó thật của teencode/emoji/thiếu dấu. |
| `human` | File `moodify_vi_human_test.csv` (cột `text,label`) do người gán nhãn - nếu bạn có. Đây mới là bằng chứng cho đồ án. |

In [ ]:
del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()
model = AutoModelForSequenceClassification.from_pretrained(best_dir).to(device)
tokenizer = AutoTokenizer.from_pretrained(best_dir)
model.eval()

eval_sets = {"test_mt": test_mt_df}
if test_vsmec_df is not None:
    eval_sets["test_vsmec_5cls"] = test_vsmec_df
if os.path.isfile(HUMAN_TEST_CSV):
    h = pd.read_csv(HUMAN_TEST_CSV, encoding="utf-8-sig")[["text", "label"]].dropna()
    h["label"] = h["label"].astype(str).str.strip().str.title().map(LABEL2ID)
    h = h.dropna(subset=["label"]); h["label"] = h["label"].astype(int)
    eval_sets["human"] = finalize(to_frame(h["text"], h["label"], "human"))
else:
    logger.info("Chưa có %s - bỏ qua tập human.", HUMAN_TEST_CSV)

summary = {}
for name, d in eval_sets.items():
    r = evaluate(model, make_loader(d, tokenizer, shuffle=False))
    summary[name] = {"n": len(d), "accuracy": r["accuracy"], "macro_f1": r["macro_f1"]}
    verdict = "ĐẠT" if r["accuracy"] >= TARGET_ACCURACY else "CHƯA ĐẠT"
    print(f"\n{'=' * 70}\n{name}  (n={len(d)})  accuracy={r['accuracy']:.2%}  macro-F1={r['macro_f1']:.4f}  -> {verdict} {TARGET_ACCURACY:.0%}")
    present = sorted(set(r["labels"]) | set(r["preds"]))
    print(classification_report(r["labels"], r["preds"], labels=present,
                                target_names=[LABELS[i] for i in present], digits=4, zero_division=0))
    print(pd.DataFrame(confusion_matrix(r["labels"], r["preds"], labels=present),
                       index=[LABELS[i] for i in present], columns=[LABELS[i] for i in present]))

with open(os.path.join(OUTPUT_DIR, "test_report.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print("\nTÓM TẮT:", json.dumps(summary, ensure_ascii=False, indent=2))

## 5. Lưu cấu hình cho service + hàm dự đoán + kiểm tra nhanh

`preprocessing.json` cùng định dạng cũ (`emoji_dict`, `teencode_dict`). `label_config.json` chứa danh sách nhãn và điểm neo V/A mới - **service và bảng mapping cảm xúc → mood trong DB phải cập nhật theo bộ nhãn 6 lớp** (thêm Love, bỏ Disgust/Other).

In [ ]:
with open(os.path.join(OUTPUT_DIR, "preprocessing.json"), "w", encoding="utf-8") as f:
    json.dump({"emoji_dict": EMOJI_DICT, "teencode_dict": TEENCODE_DICT}, f, ensure_ascii=False, indent=2)
with open(os.path.join(OUTPUT_DIR, "label_config.json"), "w", encoding="utf-8") as f:
    json.dump({"labels": LABELS, "id2label": ID2LABEL, "va_anchors": VA_ANCHORS, "model_name": MODEL_NAME,
               "max_len": MAX_LEN, "use_word_segmentation": USE_WORD_SEGMENTATION, "eval": summary},
              f, ensure_ascii=False, indent=2)

def predict_emotion(text, threshold=None):
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Văn bản dự đoán không được để trống.")
    x = preprocess_vietnamese_text(text)
    enc = tokenizer(x, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(device)
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(**enc).logits.float(), dim=-1)[0].cpu().numpy()
    k = int(np.argmax(probs))
    v = float(sum(probs[i] * VA_ANCHORS[LABELS[i]][0] for i in range(len(LABELS))))
    a = float(sum(probs[i] * VA_ANCHORS[LABELS[i]][1] for i in range(len(LABELS))))
    return {"emotion": LABELS[k], "confidence": round(float(probs[k]), 4),
            "accepted": None if threshold is None else bool(probs[k] >= threshold),
            "valence": round(v, 4), "arousal": round(a, 4),
            "probabilities": {LABELS[i]: round(float(probs[i]), 4) for i in range(len(LABELS))}}

# Câu do mình tự viết (KHÔNG phải gold set): có dấu, không dấu, teencode, emoji
SMOKE_VI = [
    ("Hôm nay mình được tăng lương, vui không tả nổi", "Enjoyment"),
    ("Cả ngày không ai nhắn tin hỏi han, thấy trống trải quá", "Sadness"),
    ("Tức quá đi mất, làm xong việc rồi mà sếp còn bắt làm lại", "Anger"),
    ("Tim đập thình thịch, nghe tiếng bước chân sau lưng mà không dám quay lại", "Fear"),
    ("Trời ơi thật không? Mình đậu rồi á? Không thể tin nổi!", "Surprise"),
    ("Anh là người em muốn ở bên cả đời", "Love"),
    ("hom nay buon qua, chang muon lam gi ca", "Sadness"),
    ("vui vcl, cuoi dau ca bung 😂", "Enjoyment"),
    ("Mệt mỏi quá, chỉ muốn khóc thôi 😭", "Sadness"),
    ("Sao họ dám đối xử với mình như vậy, điên hết cả người 😡", "Anger"),
    ("Nhớ em nhiều lắm, muốn gặp em ngay bây giờ", "Love"),
    ("Sắp thi rồi mà chưa học được gì, lo muốn chết", "Fear"),
    ("Ủa? Sao tự nhiên cô ấy lại xuất hiện ở đây vậy trời?", "Surprise"),
    ("Cuối tuần đi biển với gia đình, hạnh phúc ghê", "Enjoyment"),
    ("Đứa bạn thân chuyển đi xa rồi, nhà bỗng vắng hẳn", "Sadness"),
    ("Mình thương mẹ lắm, mẹ lo cho mình từng chút một", "Love"),
    ("dc diem 10 roi, sung suong qua di 🥳", "Enjoyment"),
    ("Đêm qua nằm mơ thấy bị đuổi, tỉnh dậy vẫn còn run", "Fear"),
    ("Đồ lừa đảo, bực không chịu được", "Anger"),
]
hit = 0
for text, gold in SMOKE_VI:
    r = predict_emotion(text)
    hit += int(r["emotion"] == gold)
    print(f"[{'OK ' if r['emotion'] == gold else 'XX '}] gold={gold:9s} pred={r['emotion']:9s} conf={r['confidence']:.2f} | {text}")
print(f"\nSmoke test: {hit}/{len(SMOKE_VI)} đúng (câu tự viết, chỉ mang tính tham khảo)")

print("\n--- Câu trung tính (không có nhãn đúng; xem confidence để chọn ngưỡng 'accepted') ---")
for text in ["Hôm nay mình ăn cơm với rau luộc", "Cuộc họp bắt đầu lúc ba giờ chiều", "Đang nằm nghe nhạc nhẹ và uống trà"]:
    r = predict_emotion(text)
    print(f"pred={r['emotion']:9s} conf={r['confidence']:.2f} V={r['valence']:+.2f} A={r['arousal']:+.2f} | {text}")
print("\nArtifact nằm tại:", OUTPUT_DIR)